# NB-16 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-16**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-16.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This moves from retained hourly outcomes to product-level predictive distributions and scores. It separates contemporaneous diagnostics from forecasts restricted to past information.

- [Volume VI, Chapter 6 — Probabilistic congestion and shadow-price forecasting](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c6)
- [Volume VI, Chapter 8 — Forecast evaluation](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c8)
- [Volume VI, Chapter 10 — FTR settlement as a target](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c10)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-16', require_key=True)
print(setup_status)


# NB-16 — Forecast the product distribution, then score it
Companion to Volume VI §§6–8,10 and Volume IV §21.5. Prerequisites: NB-06,
NB-10 and NB-15. Reuse the retained Canonical E hourly market fixture; never
regenerate its year or draw new scenarios. All 18 original paths are retained.

Separate three experiments: the saved contemporaneous-state diagnostic,
a new past-only one-hour conditional-mean forecast, and a 30-day empirical
block distribution. Hourly means alone do not specify a product distribution.
These are controlled model results, not market forecasting evidence.

In [ ]:
NOTEBOOK_ID='NB-16'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

raw=inputs['raw_inspection']
market={k:np.asarray(v) for k,v in raw['market'].items()}
arrays={k:np.asarray(v) for k,v in raw['arrays'].items()}
ftr=SimpleNamespace(KEY_LINES=raw['source_constraint_indices'],PATHS=[SimpleNamespace(**v) for v in raw['path_records']])
saved=dict(hurdle=inputs['original_contemporaneous_display_only'])
def run_saved(name,parents=(),request=None):
    entry=inputs['requests'][name]
    return execute(name,entry['method'],request if request is not None else entry['request'],parents)
h=run_saved('retained_hourly_comparison:complete_hourly')
p,e=h['prediction_result'],h['forecast_evaluation']
hourly=dict(rows=e['rows'],calibration=e['calibration'],errors=e['errors'],scaling=p['scaling'],prediction=np.asarray(p['prediction']),baseline=np.asarray(p['baseline']),truth=np.asarray(p['realized_truth']),probabilities=np.asarray(p['probabilities']),cross_sectional_mean_correlation=e['cross_sectional_mean_correlation'],topology_changes=e['topology_changes'])
b=run_saved('retained_product_blocks:complete_blocks')
p,s,e=b['predictions_and_truth'],b['evaluation_support'],b['forecast_evaluation']
product=dict(rows=[],intervals=e['intervals'],samples=np.asarray(p['samples']),weights=[np.asarray(v) for v in s['context_weights']],starts=np.asarray(p['training_block_starts']),window=s['window_days'],bandwidth=s['bandwidth'],block_variance=np.asarray(e['block_variance']),iid_variance=np.asarray(e['iid_variance']),unused_test_days=s['unused_test_days'])
for index,row in enumerate(e['rows']):
    results={}
    for method in ('forecast_scores','retained_energy_score'):
        name=method+':product_'+str(index);q=copy.deepcopy(inputs['requests'][name]['request'])
        target=q['inputs']['predictions_and_truth']
        assert target['ensemble']==target['ensemble_weights']==target['truth']==[]
        target['ensemble']=product['samples'].T.tolist()
        target['ensemble_weights']=s['uniform_weights'] if index%2==0 else s['context_weights'][index//2]
        target['truth']=p['realized_products'][index//2]
        results[method]=run_saved(name,['retained_product_blocks:complete_blocks'],q)['forecast_evaluation']
    crps,energy=results['forecast_scores'],results['retained_energy_score']
    product['rows'].append(dict(start_day=row['start_day'],end_day_exclusive=row['end_day_exclusive'],method=row['method'],mean_crps=crps['score'],coverage_80=row['coverage_80'],effective_weight_count=row['effective_weight_count'],path_crps=crps['per_row'],energy_score=energy['score']))
print('Original path order:',[p.path_id for p in ftr.PATHS])
print('Units: hourly $/MWh spread; a 1 MW obligation over 720 hourly intervals pays a $ total.')
print('Train hours: 0–5999; hourly test hours: 6000–8759. Train-only feature scaling:',hourly['scaling'])

## Activation, direction, magnitude and duration have different roles
At the start of hour t, this controlled experiment assumes the preceding
hour's load, wind, gas and topology observation is already known. Actual
publication/receipt lags are not supplied. The predictors are those t−1
observations plus the known hour-of-day sine/cosine; no t observation enters.
Three continuous scales are fitted on training hours only. There is no path
screening or test-set tuning.

Each of the four existing key constraints has positive and negative activation
channels: separate ridge logits and positive log-magnitude regressions, with a
lognormal mean correction. Fewer than five positives or negatives invokes a
training-mean fallback. These marginal models do not define a coherent joint
activation distribution. Expected signed duals map through the last observed
topology, which can be wrong on a transition. This is a one-step mean model.

Duration is retained in the historical blocks below, not estimated by the
hourly logits. Zero-event channels do not demonstrate rare-tail skill.

In [ ]:
for row in hourly['rows']:print(row)
key=list(ftr.KEY_LINES);dual=market['MU'][:,key]
directional=np.column_stack([np.maximum(dual,0),np.maximum(-dual,0)])
for j in range(directional.shape[1]):
    active=directional[:6000,j]>1e-9
    changes=np.diff(np.r_[False,active,False].astype(int));lengths=np.flatnonzero(changes==-1)-np.flatnonzero(changes==1)
    print('Direction',j,'train spells',len(lengths),'max observed hours',int(lengths.max()) if len(lengths) else 0,'boundary spells may be censored')
print('Test hours whose current topology differs from last observed:',hourly['topology_changes'])
fig,ax=plt.subplots();idx=np.arange(8)
ax.bar(idx-.18,[r['brier'] for r in hourly['rows']],.36,label='Past-only hurdle',color='#88C0D0')
ax.bar(idx+.18,[r['baseline_brier'] for r in hourly['rows']],.36,label='Train prevalence',color='#B48EAD')
ax.set(xlabel='Directional constraint channel',ylabel='Test Brier score',title='Activation scoring · include zero and rare-event support');ax.legend();plt.show()

## Calibration and temporal path errors
Compare predicted probabilities with observed frequencies in fixed width 0.2
bins. Bin counts matter; these are descriptive points, not confidence bands.
The direct-path baseline is the training mean for each hour of day. Per-path
temporal RMSE/correlation is distinct from a correlation across 18 time means.
The saved book hurdle uses realized contemporaneous load, wind, gas and true
topology and a different 70/30 split. Its summary is an information-set warning,
not a matched performance comparison with this past-only forecast.

In [ ]:
print('Saved book diagnostic information:',saved['hurdle']['information_set'])
print('Saved cross-sectional mean correlation:',saved['hurdle']['cross_sectional_correlation_of_path_time_means'])
print('New past-only cross-sectional mean correlation:',hourly['cross_sectional_mean_correlation'])
for row in hourly['errors']:print(row)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for j in range(8):
    bins=[r for r in hourly['calibration'] if r['direction']==j]
    axes[0].plot([r['probability'] for r in bins],[r['frequency'] for r in bins],marker='o',label=str(j),color=['#88C0D0','#81A1C1','#A3BE8C','#D08770','#B48EAD','#EBCB8B','#BF616A','#5E81AC'][j])
axes[0].plot([0,1],[0,1],color='#ECEFF4',linestyle=':');axes[0].set(xlabel='Mean predicted probability',ylabel='Observed frequency',title='Reliability by directional channel');axes[0].legend(ncol=4,fontsize=8)
axes[1].plot([r['calendar_rmse'] for r in hourly['errors']],label='Calendar path baseline',color='#B48EAD')
axes[1].plot([r['structured_rmse'] for r in hourly['errors']],label='Past-only hurdle',color='#88C0D0')
axes[1].set(xlabel='Original path column',ylabel='Hourly RMSE ($/MWh)',title='Time-axis errors, scored path by path');axes[1].legend();fig.tight_layout();plt.show()
print('Calibration support:',hourly['calibration'])

## Score a delivery distribution without generating new trajectories
The declared product is 30 complete 24-hour days, all hours, fixed 1 MW; there
are no weekends, DST adjustments, exchange holidays or named-market calendar
claims in this fixture. Sum the hourly payouts, never average them.

Use 220 retained overlapping training blocks starting on days 1–220 (zero-based).
Each ends before day 250. One day of preceding context is needed for weighting.
Keep each complete 720-hour multivariate trajectory intact. The baseline gives
each block equal weight. The alternative weights whole blocks by the Gaussian
distance between preceding-day load/wind/gas means and the context observed at
the product cutoff. Scales and bandwidth are training-only; bandwidth is the
median pairwise training-context distance. This is a simple conditional
historical ensemble, not samples drawn from the hourly hurdle.

Evaluate three nonoverlapping test products [250,280), [280,310), [310,340).
The training ensemble stays frozen; only observed preceding-day context updates
at each cutoff. Nonoverlap does not establish independence. The remaining 25
days cannot make a complete product and are excluded.

In [ ]:
print('Retained block count:',len(product['starts']),'training bandwidth:',product['bandwidth'],'unused test days:',product['unused_test_days'])
for row in product['rows']:print({k:v for k,v in row.items() if k!='path_crps'})
fig,axes=plt.subplots(1,2,figsize=(12,4))
for method,color in [('historical_blocks','#B48EAD'),('past_context_weighted_blocks','#88C0D0')]:
    rows=[r for r in product['rows'] if r['method']==method]
    axes[0].plot([r['start_day'] for r in rows],[r['mean_crps'] for r in rows],marker='o',label=method,color=color)
    axes[1].plot([r['start_day'] for r in rows],[r['energy_score'] for r in rows],marker='o',label=method,color=color)
axes[0].set(xlabel='Product start day',ylabel='Mean marginal CRPS ($ / 1 MW product)',title='Marginal distributions · lower is better')
axes[1].set(xlabel='Product start day',ylabel='Multivariate energy score ($ / 1 MW)',title='Joint 18-path vectors · lower is better')
axes[0].legend(fontsize=8);fig.tight_layout();plt.show()

## Predictive intervals, dependence and what failed
CRPS = Σwᵢ|xᵢ−y| − ½Σwᵢwⱼ|xᵢ−xⱼ|, evaluated for each path. The multivariate
energy score replaces absolute values with Euclidean norms on the joint
dollar payout vectors. The 10th/90th weighted empirical quantiles are predictive
intervals, not confidence intervals for forecast skill.

Whole-block weights preserve historical serial and cross-path dependence.
The weight concentration 1/Σw² is not an independent-month count: these 220
blocks overlap. An iid-day variance shortcut also misses serial dependence.
With only three evaluated products and very few rare opposite-direction events,
no reliable tail/skill confidence interval or resampling claim is made. No
bootstrap or new stochastic data was created. Context weighting loses on the
first test product; a small mean improvement elsewhere does not validate it.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4));x=np.arange(18)
for v,color in zip(product['intervals'],['#B48EAD','#88C0D0']):
    axes[0].fill_between(x,v['low'],v['high'],alpha=.25,color=color,label=v['method'])
axes[0].plot(x,product['intervals'][0]['truth'],'o',color='#ECEFF4',label='Realized')
axes[0].set(xlabel='Original path column',ylabel='$ per 1 MW product',title='First test product · empirical 80% intervals');axes[0].legend(fontsize=7)
ratio=np.divide(product['block_variance'],product['iid_variance'],out=np.full(18,np.nan),where=product['iid_variance']>0)
axes[1].bar(x,ratio,color='#D08770');axes[1].axhline(1,color='#ECEFF4',linestyle=':')
axes[1].set(xlabel='Original path column',ylabel='Historical block variance / iid-day shortcut',title='Descriptive dependence diagnostic');fig.tight_layout();plt.show()

In [ ]:
for control in inputs['negative_controls']:reject(control['method'],control['request'])
def plain_display(value):
    if isinstance(value,np.ndarray):return value.tolist()
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,dict):return {k:plain_display(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [plain_display(v) for v in value]
    return value
checked=plain_display(dict(hourly=hourly,product=product))
(session.output/'consumer-summary.json').write_text(json.dumps(checked,indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(past_only_hourly=True,complete_blocks=True,private_composed_scores=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)